In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
from s2t_fs.data.loader import load_and_prepare_data

data_params= {
    "dataset": "voxpopuli",
    "row_subsample": 1,
    "feature_subsample": 1,
    "standard_normalize": True,
    "seed": 42,
    "test_size": 0.15,
}

X_train, Y_train, X_test, Y_test, dataset_stats = load_and_prepare_data(data_params)

/opt/anaconda3/envs/s2t-fs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



In [2]:
dataset_stats

{'num_features': 1672,
 'num_total_rows': 5939,
 'train_size': 5048,
 'test_size': 891}

In [3]:
data_params =  {
    "dataset": "voxpopuli",
    "row_subsample": 1,
    "feature_subsample": 1,
    "standard_normalize": True,
    "seed": 42,
    "test_size": 0.1,
}

search_params =  {
    "seed": 42,
    "num_samples": 10, # Optuna trial sayısı
    "num_folds": 2,    # CV fold sayısı
    "test_size": 0.1,
    "refit": True
}

models_config =  {
    "AdaSTT_XGBoost": {
        "class_path": "s2t_fs.models.adastt_xgboost.AdaSTTXGBoost",
        "init_args": {}, 
        "hyperparameters": {
            "n_estimators":     [200, 50, 100],
            "learning_rate":    [0.005, 0.01, 0.05, 0.1],
            "max_depth":        [3, 5, 7, 9],
            "subsample":        [0.6, 0.8, 1.0],
            "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
            "min_child_weight": [1, 3, 7, 15],
            "reg_lambda":       [0.01, 0.1, 1.0, 10.0]
        }
    }
}


In [4]:
{"data": data_params,
"model": models_config,
"search": search_params
}

{'data': {'dataset': 'voxpopuli',
  'row_subsample': 1,
  'feature_subsample': 1,
  'standard_normalize': True,
  'seed': 42,
  'test_size': 0.1},
 'model': {'AdaSTT_XGBoost': {'class_path': 's2t_fs.models.adastt_xgboost.AdaSTTXGBoost',
   'init_args': {},
   'hyperparameters': {'n_estimators': [200, 50, 100],
    'learning_rate': [0.005, 0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.4, 0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 7, 15],
    'reg_lambda': [0.01, 0.1, 1.0, 10.0]}}},
 'search': {'seed': 42,
  'num_samples': 10,
  'num_folds': 2,
  'test_size': 0.1,
  'refit': True}}

# sectino

# new section

Sanki loguruyu temel düzeyde kurduk ve test ettik. Şimdi, farklı custom leveller ile hyperparameter tuning / experimentation sürecimizi çalıştırmayı ve loglamayı test edelim. 

In [3]:
# Otomatik yeniden yükleme (autoreload) ayarları
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import mlflow
import numpy as np

# Kendi yazdığımız "Pro" modüller
from s2t_fs.utils.logger import custom_logger as logger
from s2t_fs.data.loader import load_and_prepare_data
from s2t_fs.models.registry import prepare_models_from_config
from s2t_fs.experiment.runner import run_nested_evaluation
# from s2t_fs.utils.io import save_experiment_results # Eğer kullanıyorsanız


In [5]:
# Deney parametrelerini burada kolayca düzenleyebiliriz. İleride burası da bir yerden import edilecek. MLflow project dosyası olabilir, ya da bir başka yer olabilir. Şimdilik notebook içinde tanımlayarak başlayalım.

cfg = {
    "data_params": {
        "dataset": "voxpopuli",
        "row_subsample": 0.1,
        "feature_subsample": 0.01,
        "standard_normalize": False,
        "seed": 42,
        "test_size": 0.1,
    },
    "search_params": {
        "seed": 42,
        "num_samples": 2, # Optuna trial sayısı
        "num_folds": 3,    # CV fold sayısı
        "test_size": 0.2,
        "refit": True
    },
    "models_config": {
        "AdaSTT_XGBoost": {
            "class_path": "s2t_fs.models.adastt_xgboost.AdaSTTXGBoost",
            "init_args": {}, 
            "hyperparameters": {
                "n_estimators":     [200, 500, 1000, 2000],
                "learning_rate":    [0.005, 0.01, 0.05, 0.1],
                "max_depth":        [3, 5, 7, 9],
                "subsample":        [0.6, 0.8, 1.0],
                "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
                "min_child_weight": [1, 3, 7, 15],
                "reg_lambda":       [0.01, 0.1, 1.0, 10.0]
            }
        },
        "BoostedSDTR": {
            "class_path": "s2t_fs.models.sdtr_models.BoostedSDTR",
            "init_args": {},
            "hyperparameters": {
                "num_boosting_layers": [2, 3, 4, 5],
                "num_trees":           [10, 25, 50, 100],
                "depth":               [3, 4, 5, 6],
                "lmbda":               [0.001, 0.01, 0.1, 1.0],
                "lr":                  [5e-5, 1e-4, 5e-4, 1e-3],
                "weight_decay":        [1e-6, 1e-5, 1e-4, 1e-3]
            }
        }
    }
}

logger.info("Konfigürasyon yüklendi. Veri seti: {dataset}, Alt Örneklem: {row_subsample}", **cfg["data_params"])

Konfigürasyon yüklendi. Veri seti: voxpopuli, Alt Örneklem: 0.1


In [6]:

# s2t_fs/config.py
import warnings
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Önce logger'ı import et ve setup et (bu, tüm logging'i yakalayacak)
from s2t_fs.utils.logger import custom_logger as logger

# Optuna'nın ExperimentalWarning'i Python warnings sistemi kullanıyor
# Bu yüzden onu warnings modülü ile susturmalıyız 
from optuna.exceptions import ExperimentalWarning
env_path = find_dotenv()
load_dotenv(env_path)


warnings.filterwarnings(
    action="ignore", 
    category=ExperimentalWarning, 
    module="optuna.*"
)

warnings.filterwarnings(
    action="ignore", 
    category=FutureWarning, 
    module="xgboost.*",
    message=".*shape of the gradient and hessian.*"
)

# UserWarning'leri de filtreleyelim (Optuna bazen bunları da kullanıyor)
warnings.filterwarnings(
    action="ignore",
    category=UserWarning,
    module="optuna.*"
)

env_path = find_dotenv()
load_dotenv(env_path)

logger.info("Yapılandırma tamamlandı. Logger MLflow ve Optuna için hazır.")


Yapılandırma tamamlandı. Logger MLflow ve Optuna için hazır.


In [ ]:

from s2t_fs.utils.logger import custom_logger as logger, disable_log_category, enable_log_category

# Çok fazla detay varsa HPT loglarını gizle:
disable_log_category("HPT-Detail") 

# İşiniz bittiğinde geri açmak isterseniz:
enable_log_category("HPT-Detail")



# MLflow'da tüm bu çalışmaları gruplayacak ana klasörü (Experiment) seç/oluştur
mlflow.set_experiment("S2T_Feature_Selection_Comparisons")

# Level 1 (Top Level) Run ismini dinamik olarak belirle
run_name = f"Exp_{cfg['data_params']['dataset']}_sub{cfg['data_params']['row_subsample']}_trials{cfg['search_params']['num_samples']}"

# MLflow ana run'ını başlat
with mlflow.start_run(run_name=run_name) as parent_run:
    # logger.bind(category="Inner-Run").info(f"MLflow Parent (Inner) Run has been started. ID: {parent_run.info.run_id}.\nData Config: {cfg['data_params']}\nSearch Config: {cfg['search_params']}\n Hyperparameter Spaces: {cfg['models_config']}")

    logger.bind(category="Inner-Run").info(f"Inner Run has been started. Config: {cfg['data_params']} and {cfg['search_params']}")

    
    # 1. Sadece ana deneyi ilgilendiren üst düzey parametreleri logla
    mlflow.log_params(cfg["data_params"])
    mlflow.log_params({"global_seed": cfg["search_params"]["seed"]})
    
    # 2. Veriyi Yükle
    X_train, Y_train, X_test, Y_test, stats = load_and_prepare_data(cfg["data_params"])
    
    # 3. Modelleri Hazırla
    models = prepare_models_from_config(cfg["models_config"])
    
    # 4. Asıl Deneyi Koş (Level 2 ve Level 3 loglamaları kendi içinde otomatik hallolacak)
    margin, wers, searches = run_nested_evaluation(
        models=models, 
        X_train=X_train, Y_train=Y_train, X_test=X_test, Y_test=Y_test,
        search_params=cfg["search_params"]
    )
    
    # 5. Tüm modeller yarıştıktan sonra, bu genel deneyin özet sonuçlarını ana run'a yaz

    logger.bind(category="Inner-Run").success(f"Parent (Inner) Run completed! Gain Margin: {margin:.2f}%")

    mlflow.log_metric("final_margin_vs_baseline", margin)


Inner Run has been started. Config: {'dataset': 'voxpopuli', 'row_subsample': 0.1, 'feature_subsample': 0.01, 'standard_normalize': False, 'seed': 42, 'test_size': 0.1} and {'seed': 42, 'num_samples': 2, 'num_folds': 3, 'test_size': 0.2, 'refit': True}
Hyperparameter tuning of AdaSTT_XGBoost: (2 trials)
Hyperparameter tuning of BoostedSDTR: (2 trials)
Parent (Inner) Run completed! Gain Margin: 0.46%


In [8]:
run_name

'Exp_voxpopuli_sub0.1_trials2'